# Comparaison CPU float vs CPU FPGA-compatible

Notebook minimal pour comparer les deux méthodes CPU sur les mêmes signaux et décider laquelle garder pour la campagne complète.

In [13]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

EPS = 1e-12
DDS_LUT_SIZE = 1024
DDS_PHASE_BITS = 32
DDS_LUT_BITS = 10

HEADER_RE = re.compile(
    r"PRN=(?P<prn>-?\d+)"
    r".*SNR=(?P<snr>-?\d+(?:\.\d+)?)dB"
    r".*Doppler=(?P<doppler>-?\d+(?:\.\d+)?)Hz"
    r".*CA_SHIFT=(?P<ca>-?\d+)chips",
    re.IGNORECASE,
)


def _extract_lut_block(text, lut_name):
    pattern = rf"{lut_name}\[DDS_LUT_SIZE\]\s*=\s*\{{(.*?)\}};"
    m = re.search(pattern, text, flags=re.S)
    if not m:
        raise ValueError(f"Table {lut_name} introuvable dans dds_lut_rom.h")
    block = m.group(1)

    values = []
    for v in re.findall(r"\(osc_t\)\s*([-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?)", block):
        values.append(float(v))

    arr = np.asarray(values, dtype=np.float32)
    if arr.size != DDS_LUT_SIZE:
        raise ValueError(f"{lut_name} contient {arr.size} elements, attendu {DDS_LUT_SIZE}")
    return arr


def load_dds_luts_from_header(header_file_path):
    p = Path(header_file_path)
    if not p.exists():
        candidates = [
            Path("dds_lut_rom.h"),
            Path("../dds_lut_rom.h"),
            Path("../../STSA/dds_lut_rom.h"),
        ]
        p = next((c for c in candidates if c.exists()), p)
    if not p.exists():
        raise FileNotFoundError(f"dds_lut_rom.h introuvable: {header_file_path}")

    content = p.read_text(encoding="utf-8", errors="ignore")
    dds_sin_lut = _extract_lut_block(content, "DDS_SIN_LUT")
    dds_cos_lut = _extract_lut_block(content, "DDS_COS_LUT")
    return dds_sin_lut, dds_cos_lut


def parse_meta_from_header(csv_file):
    csv_file = Path(csv_file)
    with csv_file.open("r", encoding="utf-8", errors="ignore") as handle:
        first = handle.readline().strip()
    match = HEADER_RE.search(first)
    if not match:
        raise ValueError(f"Metadonnees introuvables dans {csv_file.name}")
    return {
        "prn": int(match.group("prn")),
        "snr": float(match.group("snr")),
        "doppler": int(round(float(match.group("doppler")))),
        "ca_shift": int(match.group("ca")),
    }


def load_signal_pm1(csv_file):
    csv_file = Path(csv_file)
    with csv_file.open("r", encoding="utf-8", errors="ignore") as handle:
        line1 = handle.readline().strip()
        line2 = handle.readline().strip()
    if line1 and not line1.startswith("#") and not line2:
        line2 = line1
    if not line2:
        raise ValueError(f"Signal vide: {csv_file}")
    data = np.fromstring(line2, sep=",", dtype=np.float32)
    if data.size == 0:
        raise ValueError(f"Format invalide: {csv_file}")
    data = np.where(data == 255, -1, data).astype(np.float32)
    return data


def load_prn_sequence(prn_dir, prn, n_samples):
    prn_dir = Path(prn_dir)
    candidates = [
        prn_dir / f"PRN-{prn}.csv",
        prn_dir / f"prn-{prn}.csv",
        prn_dir / f"PRN_{prn}.csv",
        prn_dir / f"prn_{prn}.csv",
    ]
    prn_path = next((path for path in candidates if path.exists()), None)
    if prn_path is None:
        raise FileNotFoundError(f"PRN {prn} introuvable dans {prn_dir}")
    seq = np.genfromtxt(prn_path, delimiter=",", dtype=np.float64)
    seq = np.atleast_1d(seq).reshape(-1)
    if seq.size < n_samples:
        raise ValueError(f"PRN trop court ({seq.size}) dans {prn_path}, attendu >= {n_samples}")
    return seq[:n_samples].astype(np.float32)


def collect_signal_files(signals_dir, signal_glob):
    signals_dir = Path(signals_dir)
    files = sorted(signals_dir.glob(signal_glob)) if signals_dir.exists() else []
    if not files and signals_dir.exists():
        files = sorted(signals_dir.glob("*.csv"))
    valid_files = []
    for file_path in files:
        try:
            parse_meta_from_header(file_path)
            valid_files.append(file_path)
        except Exception:
            continue
    return valid_files


def read_csv_ints(path):
    txt = Path(path).read_text(encoding="utf-8", errors="ignore")
    vals = []
    for token in re.split(r"[\s,;]+", txt):
        if not token:
            continue
        try:
            vals.append(int(token))
        except ValueError:
            pass
    return np.asarray(vals, dtype=np.int64)


def _precompute_prn_shifts(prn, n_samples, nb_phases):
    shifts = np.floor(np.arange(nb_phases) * n_samples / nb_phases).astype(int)
    prn_shifted = np.empty((nb_phases, n_samples), dtype=np.float32)
    for tau in range(nb_phases):
        prn_shifted[tau] = np.roll(prn, -shifts[tau])
    return prn_shifted


def cpu_float_acquisition(signal, prn, cfg):
    n_samples = int(cfg["n_samples"])
    nb_phases = int(cfg["nb_phases"])
    fs_hz = float(cfg["fs_hz"])
    if_hz = float(cfg["if_hz"])
    fd_start = int(cfg["fd_start"])
    fd_end = int(cfg["fd_end"])
    fd_step = int(cfg["fd_step"])
    ratio_min = float(cfg["detection_ratio_min"])

    signal = np.asarray(signal[:n_samples], dtype=np.complex64)
    prn = np.asarray(prn[:n_samples], dtype=np.float32)
    prn_shifted = _precompute_prn_shifts(prn, n_samples, nb_phases)
    doppler_range = np.arange(fd_start, fd_end + 1, fd_step, dtype=np.int32)
    corr_matrix = np.zeros((len(doppler_range), nb_phases), dtype=np.float64)
    n = np.arange(n_samples, dtype=np.float64)
    ts = 1.0 / fs_hz

    t0 = time.perf_counter()
    for k, fd in enumerate(doppler_range):
        carrier = np.exp(-1j * 2.0 * np.pi * (if_hz + float(fd)) * n * ts)
        corr = prn_shifted @ (signal * carrier)
        corr_matrix[k, :] = np.abs(corr) ** 2
    t1 = time.perf_counter()

    row, col = np.unravel_index(int(np.argmax(corr_matrix)), corr_matrix.shape)
    peak = float(corr_matrix[row, col])
    mean_corr = float(np.mean(corr_matrix))
    peak_ratio = peak / mean_corr if mean_corr > EPS else 0.0
    return {
        "method": "cpu_float",
        "detected": int(peak_ratio >= ratio_min),
        "doppler": int(doppler_range[row]),
        "phase": int(col),
        "peak": peak,
        "peak_ratio": peak_ratio,
        "time_s": float(t1 - t0),
    }


def _ap_int2_wrap(x):
    x_int = np.rint(np.real(x)).astype(np.int64)
    x_int = x_int & 0x3
    return np.where(x_int >= 2, x_int - 4, x_int).astype(np.int8)


def cpu_lut_angles_acquisition(signal, prn, cfg, dds_sin_lut, dds_cos_lut):
    """Version float rapide: seule la porteuse utilise la LUT DDS."""
    n_samples = int(cfg["n_samples"])
    nb_phases = int(cfg["nb_phases"])
    fs_hz = float(cfg["fs_hz"])
    if_hz = float(cfg["if_hz"])
    fd_start = int(cfg["fd_start"])
    fd_end = int(cfg["fd_end"])
    fd_step = int(cfg["fd_step"])
    ratio_min = float(cfg["detection_ratio_min"])

    if fd_step <= 0:
        fd_step = 1

    signal = np.asarray(signal[:n_samples], dtype=np.complex64)
    prn = np.asarray(prn[:n_samples], dtype=np.float32)
    prn_shifted = _precompute_prn_shifts(prn, n_samples, nb_phases)

    if fd_start <= fd_end:
        doppler_range = np.arange(fd_start, fd_end + 1, fd_step, dtype=np.int32)
    else:
        doppler_range = np.arange(fd_start, fd_end - 1, -fd_step, dtype=np.int32)
    corr_matrix = np.zeros((len(doppler_range), nb_phases), dtype=np.float64)

    t0 = time.perf_counter()
    n_idx = np.arange(n_samples, dtype=np.uint64)
    shift = DDS_PHASE_BITS - DDS_LUT_BITS

    for k, fd in enumerate(doppler_range):
        scale = 1 << DDS_PHASE_BITS
        num = int(-(if_hz + float(fd)) * scale)
        den = int(fs_hz)
        if num >= 0:
            num += den // 2
        else:
            num -= den // 2
        phase_inc = int(num // den) & 0xFFFFFFFF

        phase_acc = (n_idx * np.uint64(phase_inc)) & np.uint64(0xFFFFFFFF)
        lut_idx = ((phase_acc >> np.uint64(shift)) & np.uint64(DDS_LUT_SIZE - 1)).astype(np.int64)

        c = dds_cos_lut[lut_idx].astype(np.float64, copy=False)
        s = dds_sin_lut[lut_idx].astype(np.float64, copy=False)
        data_mix = signal.astype(np.float64, copy=False) * (c - 1j * s)

        corr = prn_shifted @ data_mix
        corr_matrix[k, :] = np.abs(corr) ** 2

    t1 = time.perf_counter()

    row, col = np.unravel_index(int(np.argmax(corr_matrix)), corr_matrix.shape)
    peak = float(corr_matrix[row, col])
    mean_corr = float(np.mean(corr_matrix))
    peak_ratio = peak / mean_corr if mean_corr > EPS else 0.0
    return {
        "method": "cpu_lut_angles",
        "detected": int(peak_ratio >= ratio_min),
        "doppler": int(doppler_range[row]),
        "phase": int(col),
        "peak": peak,
        "peak_ratio": peak_ratio,
        "time_s": float(t1 - t0),
    }


CFG = {
    "signals_dir": "signals",
    "prn_dir": "PRN",
    "signal_glob": "*.csv",
    "limit_signals": 50,
    "use_tb_single_files": True,
    "tb_signal_candidates": ["signal.csv", "../signal.csv", "../../STSA/signal.csv"],
    "tb_prn_candidates": ["PRN-25.csv", "../PRN-25.csv", "../../STSA/PRN-25.csv"],
    "tb_default_prn": 25,
    "fs_hz": 11_999_000.0,
    "if_hz": 3_563_000.0,
    "n_samples": 11999,
    "nb_phases": 1023,
    "fd_start": -10000,
    "fd_end": 10000,
    "fd_step": 250,
    "detection_ratio_min": 2.5,
    "dds_header": "dds_lut_rom.h",  # fallback auto: ../ et ../../STSA
    "compare_export_csv": "Resultats validation SA/comparaison_cpu_float_vs_lut_angles.csv",
    "quick_mode": False,
    "quick_n_samples": 4092,
    "quick_nb_phases": 511,
    "quick_fd_step": 1250,
}

print(pd.Series(CFG, dtype="object"))

signals_dir                                                       signals
prn_dir                                                               PRN
signal_glob                                                         *.csv
limit_signals                                                          50
use_tb_single_files                                                  True
tb_signal_candidates    [signal.csv, ../signal.csv, ../../STSA/signal....
tb_prn_candidates       [PRN-25.csv, ../PRN-25.csv, ../../STSA/PRN-25....
tb_default_prn                                                         25
fs_hz                                                          11999000.0
if_hz                                                           3563000.0
n_samples                                                           11999
nb_phases                                                            1023
fd_start                                                           -10000
fd_end                                

In [14]:
dds_sin_lut, dds_cos_lut = load_dds_luts_from_header(CFG["dds_header"])
rows = []

cfg_run = dict(CFG)
if bool(cfg_run.get("quick_mode", False)):
    cfg_run["n_samples"] = min(int(cfg_run["n_samples"]), int(cfg_run.get("quick_n_samples", 4092)))
    cfg_run["nb_phases"] = min(int(cfg_run["nb_phases"]), int(cfg_run.get("quick_nb_phases", 511)))
    cfg_run["fd_step"] = max(int(cfg_run["fd_step"]), int(cfg_run.get("quick_fd_step", 1250)))
    print(
        "[MODE RAPIDE] "
        f"n_samples={cfg_run['n_samples']}, nb_phases={cfg_run['nb_phases']}, fd_step={cfg_run['fd_step']}"
    )
else:
    print(
        "[MODE COMPLET] "
        f"n_samples={cfg_run['n_samples']}, nb_phases={cfg_run['nb_phases']}, fd_step={cfg_run['fd_step']}"
    )

if bool(cfg_run.get("use_tb_single_files")):
    signal_path = next((Path(p) for p in cfg_run["tb_signal_candidates"] if Path(p).exists()), None)
    prn_path = next((Path(p) for p in cfg_run["tb_prn_candidates"] if Path(p).exists()), None)
    if signal_path is None or prn_path is None:
        raise FileNotFoundError("Impossible de trouver signal.csv et/ou PRN-25.csv")

    rx_all = read_csv_ints(signal_path)
    prn_all = read_csv_ints(prn_path)
    n_samples = int(cfg_run["n_samples"])
    if rx_all.size < n_samples or prn_all.size < n_samples:
        raise ValueError(
            f"CSV trop court: signal={rx_all.size}, prn={prn_all.size}, attendu >= {n_samples}"
        )

    signal = rx_all[:n_samples].astype(np.float32).astype(np.complex64)
    prn_seq = prn_all[:n_samples].astype(np.float32)
    prn = int(cfg_run.get("tb_default_prn", 25))

    r_float = cpu_float_acquisition(signal, prn_seq, cfg_run)
    r_lut = cpu_lut_angles_acquisition(signal, prn_seq, cfg_run, dds_sin_lut, dds_cos_lut)

    rows.append({
        "file": str(signal_path),
        "prn": prn,
        "snr_db": np.nan,
        "doppler_true_hz": np.nan,
        "float_detected": int(r_float["detected"]),
        "lut_detected": int(r_lut["detected"]),
        "same_detected": int(int(r_float["detected"]) == int(r_lut["detected"])),
        "float_doppler_hz": int(r_float["doppler"]),
        "lut_doppler_hz": int(r_lut["doppler"]),
        "doppler_delta_hz": int(abs(int(r_float["doppler"]) - int(r_lut["doppler"]))),
        "float_phase": int(r_float["phase"]),
        "lut_phase": int(r_lut["phase"]),
        "phase_delta": int(abs(int(r_float["phase"]) - int(r_lut["phase"]))),
        "float_peak": float(r_float["peak"]),
        "lut_peak": float(r_lut["peak"]),
        "peak_delta": float(abs(float(r_float["peak"]) - float(r_lut["peak"]))),
        "float_peak_ratio": float(r_float["peak_ratio"]),
        "lut_peak_ratio": float(r_lut["peak_ratio"]),
        "peak_ratio_delta": float(abs(float(r_float["peak_ratio"]) - float(r_lut["peak_ratio"]))),
        "float_time_s": float(r_float["time_s"]),
        "lut_time_s": float(r_lut["time_s"]),
    })
    print(f"[1/1] {signal_path} avec {prn_path}")
else:
    signals_dir = Path(CFG["signals_dir"])
    prn_dir = Path(CFG["prn_dir"])
    if not signals_dir.exists():
        raise FileNotFoundError(f"Dossier signaux introuvable: {signals_dir}")
    if not prn_dir.exists():
        raise FileNotFoundError(f"Dossier PRN introuvable: {prn_dir}")

    signal_files = collect_signal_files(signals_dir, CFG["signal_glob"])
    if not signal_files:
        raise RuntimeError("Aucun signal valide trouve")
    if CFG.get("limit_signals"):
        signal_files = signal_files[: int(CFG["limit_signals"])]

    print(f"Signaux compares: {len(signal_files)}")
    prn_cache = {}

    for i, sig_file in enumerate(signal_files, 1):
        meta = parse_meta_from_header(sig_file)
        prn = int(meta["prn"])
        signal = load_signal_pm1(sig_file)
        n_samples = int(cfg_run["n_samples"])
        if signal.size < n_samples:
            signal = np.pad(signal, (0, n_samples - signal.size), mode="edge")
        else:
            signal = signal[:n_samples]
        signal = signal.astype(np.complex64)

        if prn not in prn_cache:
            prn_cache[prn] = load_prn_sequence(prn_dir, prn, n_samples)
        prn_seq = prn_cache[prn]

        r_float = cpu_float_acquisition(signal, prn_seq, cfg_run)
        r_lut = cpu_lut_angles_acquisition(signal, prn_seq, cfg_run, dds_sin_lut, dds_cos_lut)

        rows.append({
            "file": sig_file.name,
            "prn": prn,
            "snr_db": float(meta["snr"]),
            "doppler_true_hz": int(meta["doppler"]),
            "float_detected": int(r_float["detected"]),
            "lut_detected": int(r_lut["detected"]),
            "same_detected": int(int(r_float["detected"]) == int(r_lut["detected"])),
            "float_doppler_hz": int(r_float["doppler"]),
            "lut_doppler_hz": int(r_lut["doppler"]),
            "doppler_delta_hz": int(abs(int(r_float["doppler"]) - int(r_lut["doppler"]))),
            "float_phase": int(r_float["phase"]),
            "lut_phase": int(r_lut["phase"]),
            "phase_delta": int(abs(int(r_float["phase"]) - int(r_lut["phase"]))),
            "float_peak": float(r_float["peak"]),
            "lut_peak": float(r_lut["peak"]),
            "peak_delta": float(abs(float(r_float["peak"]) - float(r_lut["peak"]))),
            "float_peak_ratio": float(r_float["peak_ratio"]),
            "lut_peak_ratio": float(r_lut["peak_ratio"]),
            "peak_ratio_delta": float(abs(float(r_float["peak_ratio"]) - float(r_lut["peak_ratio"]))),
            "float_time_s": float(r_float["time_s"]),
            "lut_time_s": float(r_lut["time_s"]),
        })
        print(f"[{i}/{len(signal_files)}] {sig_file.name}")

df_cmp = pd.DataFrame(rows)
if df_cmp.empty:
    raise RuntimeError("Aucun resultat de comparaison")

summary = pd.DataFrame([{
    "n_signals": int(len(df_cmp)),
    "decision_agreement_pct": float(100.0 * df_cmp["same_detected"].mean()),
    "doppler_exact_agreement_pct": float(100.0 * (df_cmp["doppler_delta_hz"] == 0).mean()),
    "phase_exact_agreement_pct": float(100.0 * (df_cmp["phase_delta"] == 0).mean()),
    "doppler_delta_mean_hz": float(df_cmp["doppler_delta_hz"].mean()),
    "phase_delta_mean": float(df_cmp["phase_delta"].mean()),
    "peak_ratio_delta_mean": float(df_cmp["peak_ratio_delta"].mean()),
    "float_time_mean_s": float(df_cmp["float_time_s"].mean()),
    "lut_angles_time_mean_s": float(df_cmp["lut_time_s"].mean()),
}])

out_csv = Path(CFG["compare_export_csv"])
out_csv.parent.mkdir(parents=True, exist_ok=True)
df_cmp.to_csv(out_csv, index=False)

print("\n=== Resume comparaison ===")
display(summary.round(6))
print(f"\nCSV detail: {out_csv}")

[MODE COMPLET] n_samples=11999, nb_phases=1023, fd_step=250


/tmp/ipykernel_2264/80618599.py:236: ComplexWarning: Casting complex values to real discards the imaginary part
  data_mix = signal.astype(np.float64, copy=False) * (c - 1j * s)


[1/1] ../signal.csv avec ../PRN-25.csv

=== Resume comparaison ===


,n_signals,decision_agreement_pct,doppler_exact_agreement_pct,phase_exact_agreement_pct,doppler_delta_mean_hz,phase_delta_mean,peak_ratio_delta_mean,float_time_mean_s,lut_angles_time_mean_s
0,1,100.0,100.0,100.0,0.0,0.0,0.015469,109.296684,108.149886



CSV detail: Resultats validation SA/comparaison_cpu_float_vs_lut_angles.csv


In [15]:
mismatch = df_cmp[(df_cmp["same_detected"] == 0) | (df_cmp["doppler_delta_hz"] != 0) | (df_cmp["phase_delta"] != 0)].copy()
print(f"Nombre de cas differents: {len(mismatch)}")
display(mismatch.head(20))

if len(mismatch) == 0:
    print("Les deux methodes sont identiques sur l'echantillon teste (decision, Doppler, phase).")
else:
    print("Des ecarts existent. Regarder le tableau ci-dessus pour choisir la methode la plus robuste/rapide.")

Nombre de cas differents: 0


,file,prn,snr_db,doppler_true_hz,float_detected,lut_detected,same_detected,float_doppler_hz,lut_doppler_hz,doppler_delta_hz,...,lut_phase,phase_delta,float_peak,lut_peak,peak_delta,float_peak_ratio,lut_peak_ratio,peak_ratio_delta,float_time_s,lut_time_s


Les deux methodes sont identiques sur l'echantillon teste (decision, Doppler, phase).


In [16]:
# ==========================================================
# CELLULE 4 - TABLEAU SIMPLE POUR LE RAPPORT
# Justification de coherence et choix de la methode LUT
# ==========================================================
import numpy as np
import pandas as pd

if "df_cmp" not in globals() or not isinstance(df_cmp, pd.DataFrame) or df_cmp.empty:
    raise RuntimeError("Lance d'abord la cellule de comparaison pour generer df_cmp.")

n = int(len(df_cmp))

coherence_table = pd.DataFrame([
    {
        "n_signaux": n,
        "accord_decision_pct": float(100.0 * df_cmp["same_detected"].mean()),
        "accord_doppler_exact_pct": float(100.0 * (df_cmp["doppler_delta_hz"] == 0).mean()),
        "accord_phase_exact_pct": float(100.0 * (df_cmp["phase_delta"] == 0).mean()),
    }
])

valeurs_table = pd.DataFrame([
    {
        "doppler_float_moy_hz": float(df_cmp["float_doppler_hz"].mean()),
        "doppler_lut_moy_hz": float(df_cmp["lut_doppler_hz"].mean()),
        "delta_doppler_moyen_hz": float(df_cmp["doppler_delta_hz"].mean()),
        "phase_float_moy": float(df_cmp["float_phase"].mean()),
        "phase_lut_moy": float(df_cmp["lut_phase"].mean()),
        "delta_phase_moyen_chip": float(df_cmp["phase_delta"].mean()),
        "peak_float_moy": float(df_cmp["float_peak"].mean()),
        "peak_lut_moy": float(df_cmp["lut_peak"].mean()),
        "delta_peak_moyen": float(df_cmp["peak_delta"].mean()),
        "delta_peak_ratio_moyen": float(df_cmp["peak_ratio_delta"].mean()),
    }
])

temps_table = pd.DataFrame([
    {
        "temps_moyen_cpu_float_s": float(df_cmp["float_time_s"].mean()),
        "temps_moyen_cpu_lut_s": float(df_cmp["lut_time_s"].mean()),
        "ratio_temps_lut_vs_float": float(df_cmp["lut_time_s"].mean() / max(df_cmp["float_time_s"].mean(), 1e-12)),
    }
])

# Critere simple pour la conclusion rapport
is_coherent = bool(
    float(coherence_table.loc[0, "accord_decision_pct"]) == 100.0
    and float(coherence_table.loc[0, "accord_doppler_exact_pct"]) == 100.0
    and float(coherence_table.loc[0, "accord_phase_exact_pct"]) == 100.0
)

recommendation = pd.DataFrame([
    {
        "coherence_globale": "OK" if is_coherent else "A verifier",
        "methode_recommandee_pour_comparaison_IP": "cpu_lut_angles",
        "justification": (
            "Resultats coherents avec cpu_float sur decision, Doppler et phase"
            if is_coherent
            else "Des ecarts existent: analyser les cas mismatch avant conclusion"
        ),
    }
])

print("=== TABLEAU 1 - COHERENCE ===")
display(coherence_table.round(6))

print("\n=== TABLEAU 2 - VALEURS DOPPLER / PHASE / PEAK ===")
display(valeurs_table.round(6))

print("\n=== TABLEAU 3 - TEMPS DE CALCUL ===")
display(temps_table.round(6))

print("\n=== CONCLUSION PROPOSEE ===")
display(recommendation)

# Exports simples pour insertion rapport
out_dir = Path("Resultats validation SA")
out_dir.mkdir(parents=True, exist_ok=True)
out_coherence = out_dir / "tableau_coherence_float_vs_lut.csv"
out_valeurs = out_dir / "tableau_valeurs_float_vs_lut.csv"
out_temps = out_dir / "tableau_temps_float_vs_lut.csv"

coherence_table.to_csv(out_coherence, index=False)
valeurs_table.to_csv(out_valeurs, index=False)
temps_table.to_csv(out_temps, index=False)

print("\nExports CSV rapport:")
print(f"- {out_coherence}")
print(f"- {out_valeurs}")
print(f"- {out_temps}")

=== TABLEAU 1 - COHERENCE ===


,n_signaux,accord_decision_pct,accord_doppler_exact_pct,accord_phase_exact_pct
0,1,100.0,100.0,100.0



=== TABLEAU 2 - VALEURS DOPPLER / PHASE / PEAK ===


,doppler_float_moy_hz,doppler_lut_moy_hz,delta_doppler_moyen_hz,phase_float_moy,phase_lut_moy,delta_phase_moyen_chip,peak_float_moy,peak_lut_moy,delta_peak_moyen,delta_peak_ratio_moyen
0,-1750.0,-1750.0,0.0,150.0,150.0,0.0,2.028802e+06,2.029162e+06,360.195153,0.015469



=== TABLEAU 3 - TEMPS DE CALCUL ===


,temps_moyen_cpu_float_s,temps_moyen_cpu_lut_s,ratio_temps_lut_vs_float
0,109.296684,108.149886,0.989507



=== CONCLUSION PROPOSEE ===


,coherence_globale,methode_recommandee_pour_comparaison_IP,justification
0,OK,cpu_lut_angles,Resultats coherents avec cpu_float sur decisio...



Exports CSV rapport:
- Resultats validation SA/tableau_coherence_float_vs_lut.csv
- Resultats validation SA/tableau_valeurs_float_vs_lut.csv
- Resultats validation SA/tableau_temps_float_vs_lut.csv
